In [1]:
kx_input = """
ArchiveEntry "IJRR17/Corollary 1.1: Velocity-controlled passive safety"

Definitions
  Real ep;      /* time limit for control decisions */
  Real W;       /* maximum steering */
  Real V;       /* maximum obstacle velocity */

  Real stopDist(Real v) = 0;
  Real accelComp(Real v) = ep*(v+V);
  Real admissibleSeparation(Real v) = stopDist(v) + accelComp(v);

  Bool isWellformedDir(Real dx, Real dy) <-> dx^2 + dy^2 = 1;                        /* The orientation of the robot is a unit vector. */

  Bool bounds() <-> (                                                                     /* Bounds for global constants */
      ep > 0                                                                            /* Controller reaction time */
    & V >= 0
  );

  /* Stopped somewhere safe initially */
  Bool initialState(Real x, Real y, Real v, Real dx, Real dy, Real xo, Real yo) <-> (
    v = 0
    & (x-xo)^2 + (y-yo)^2 > 0
    & isWellformedDir(dx,dy)
  );

  /* Under these assumptions we guarantee safety */
  Bool assumptions(Real x, Real y, Real v, Real dx, Real dy, Real xo, Real yo) <->
    bounds() & initialState(x,y,v,dx,dy,xo,yo);

  /* Conditions that are true on each loop iteration */
  Bool loopinv(Real x, Real y, Real v, Real dx, Real dy, Real xo, Real yo) <-> (
    v >= 0
    & isWellformedDir(dx,dy)
    & (v>0 -> abs(x-xo) > stopDist(v) | abs(y-yo) > stopDist(v))
  );

  HP controlFunc ::= {
    /* obstacle control */
    {
        vxo := *; vyo := *;
        ?vxo^2+vyo^2<=V^2;
    }
    ++
    /* robot control */
    {
        /* brake on current curve or remain stopped */
        { v := 0; w := 0; }
        ++
        /* or choose a new safe curve */
        { v := *; ?0<=v;
        w := *; ?-W<=w & w<=W;       /* choose steering */
        r := *;
        xo := *; yo := *;            /* measure closest obstacle on the curve */

        /* admissible curve */
        ?r!=0 & r*w = v;

        /* use that curve, if it is a safe one (admissible velocities) */
        ? abs(x-xo) > admissibleSeparation(v)
        | abs(y-yo) > admissibleSeparation(v);
        }
      }
  }

    HP stepFunc ::= { 
        x' = v * dx, y' = v * dy,                /* move */
        dx' = -w * dy, dy' = w * dx,             /* follow curve */
        xo' = vxo, yo' = vyo,                    /* obstacle moves */
        t' = 1 & t <= ep & v >= 0
    }
End.

ProgramVariables
  Real x;    /* robot position: x */
  Real y;    /* robot position: y */
  Real v;    /* robot translational velocity */
  Real a;    /* robot translational acceleration */
  Real dx;   /* robot orientation: x */
  Real dy;   /* robot orientation: y */
  Real w;    /* robot rotational velocity */
  Real r;    /* robot curve radius */
  Real xo;   /* position of closest obstacle on curve */
  Real yo;
  Real vxo;  /* velocity vector of obstacle */
  Real vyo;
  Real t;    /* time */
End.

Problem
assumptions(x, y, v, dx, dy, xo, yo) ->
  [
    { 
      {
        {
          controlFunc;
        };
        t := 0;
      }

      /* dynamics */
      {
        stepFunc;@invariant(t>=0, isWellformedDir(dx,dy),
          (-t*V <= xo - old(xo) & xo - old(xo) <= t*V),
          (-t*V <= yo - old(yo) & yo - old(yo) <= t*V),
          (x'=0*dx -> x = old(x)),
          (x'=0*dx -> y = old(y)),
          (x'=v*dx -> -t * v <= x - old(x) & x - old(x) <= t * v),
          (x'=v*dx -> -t * v <= y - old(y) & y - old(y) <= t * v)
        )
      }
    }*@invariant(loopinv(x, y, v, dx, dy, xo, yo))
  ](v>0 -> (x - xo)^2 + (y - yo)^2 > 0)
End.
"""

In [3]:
# PyAST_test.py
from keymaerax_parser import KeYmaeraXParser, convert_keymaerax_ast_to_intermediate_pyast
from gym_generator import generate_gym_environment_code # Import the generator

# --- KeYmaera X Input ---


# --- Main Execution ---
if __name__ == "__main__":
    print("--- Parsing KeYmaera X Input into internal AST ---")
    parser = KeYmaeraXParser(kx_input)
    kx_ast = parser.parse()
    # Print the internal KeYmaera X AST for debugging/verification
    # print(kx_ast) # Commented out for brevity

    print("\n--- Converting KeYmaera X AST to Intermediate Python AST ---")
    py_ast_config = convert_keymaerax_ast_to_intermediate_pyast(kx_ast)
    print("Intermediate Python AST (GymEnvironmentConfig):\n", py_ast_config)

    print("\n--- Generating Gymnasium environment code ---")
    generated_code = generate_gym_environment_code(py_ast_config)

    # print(generated_code)
    output_filename = "generated_kx_env.py"
    with open(output_filename, "w") as f:
        f.write(generated_code)

    print(f"Generated Gymnasium environment code saved to {output_filename}")

#     print("\n--- Instructions to use the generated environment ---")
#     print(f"""
# To use the generated environment, ensure `ast_config.py`, `keymaerax_parser.py`, and `gym_generator.py`
# are in the same directory as this script. Then:

# 1.  Run `PyAST_test.py`. This will generate `generated_kx_env.py`.
# 2.  You can then import and use the generated environment in another Python script or interactive session:

# ```python
# import gymnasium as gym
# # The class name is derived from the Theorem name: "IJRR17/Theorem 1: Static safety"
# # spaces, slashes, colons, etc., are replaced with underscores.
# from generated_kx_env import IJRR17_Theorem_1_Static_safety 
# import numpy as np

# # Instantiate the environment
# env = IJRR17_Theorem_1_Static_safety()

# observation, info = env.reset()
# print("Initial observation:", observation)
# print("Observation Space:", env.observation_space)
# print("Action Space:", env.action_space)

# # The action space for this KeYmaera X example's controlFunc is a Dict.
# # It represents a top-level choice between 3 branches, and some branches
# # involve nondeterministic assignments (Box spaces).

# # Action Space Example Structure (simplified):
# # Dict(
# #   'choice_ChoiceLevel_0': Discrete(3),  # 0, 1, or 2 for the three branches
# #   'assign_w': Box(low=-inf, high=inf, shape=()), # For 'w := *' in branch 2
# #   'assign_r': Box(low=-inf, high=inf, shape=()), # For 'r := *' in branch 2
# #   'assign_xo': Box(low=-inf, high=inf, shape=()), # For 'xo := *' in branch 2
# #   'assign_yo': Box(low=-inf, high=inf, shape=()), # For 'yo := *' in branch 2
# # )

# # When using a Dict action space, you must provide values for *all* keys,
# # even if a chosen branch doesn't explicitly use a particular nondeterministic assignment.
# # For unused Box spaces, you can provide a dummy value (e.g., 0.0).

# # Let's assume some constant values for demonstration (these are set internally
# # by the generated environment, but we need to know their effect on 'a'):
# # b = some_positive_value (e.g., 1.0)
# # A = some_positive_value (e.g., 1.0)
# # W = some_positive_value (e.g., 1.0)

# # Example 1: Choose the first branch: {{ a := -b; }} (index 0)
# action_branch_0 = {{
#     'choice_ChoiceLevel_0': 0, # Select the first branch
#     'assign_w': 0.0,           # Dummy value for unused nondeterministic assignments
#     'assign_r': 0.0,
#     'assign_xo': 0.0,
#     'assign_yo': 0.0,
# }}

# print("\\nTaking 5 steps with action_branch_0 (a := -b;):")
# for i in range(5):
#     observation, reward, terminated, truncated, info = env.step(action_branch_0)
#     print(f"Step {{i+1}}: Obs={{observation}}, Reward={{reward:.2f}}, Terminated={{terminated}}, Truncated={{truncated}}")
#     if terminated or truncated:
#         print("Episode finished.")
#         break
# env.close()

# # Example 2: Reset and choose the third branch: {{ a := A; w := *; ... }} (index 2)
# # This branch requires specific values for w, r, xo, yo.
# print("\\nResetting environment and taking 5 steps with action_branch_2 (a := A; w := *; etc.):")
# observation, info = env.reset()

# action_branch_2 = {{
#     'choice_ChoiceLevel_0': 2, # Select the third branch
#     'assign_w': 0.5,           # Example value for w (within -W to W, assuming W=1)
#     'assign_r': 1.0,           # Example value for r (non-zero)
#     'assign_xo': 10.0,         # Example value for xo
#     'assign_yo': 0.0,          # Example value for yo
# }}

# for i in range(5):
#     observation, reward, terminated, truncated, info = env.step(action_branch_2)
#     print(f"Step {{i+1}}: Obs={{observation}}, Reward={{reward:.2f}}, Terminated={{terminated}}, Truncated={{truncated}}")
#     if terminated or truncated:
#         print("Episode finished.")
#         break
# env.close()
# """)

--- Parsing KeYmaera X Input into internal AST ---

--- Converting KeYmaera X AST to Intermediate Python AST ---
Intermediate Python AST (GymEnvironmentConfig):
 [GymEnvironmentConfig](IJRR17_Corollary_1_1__Velocity-controlled_passive_safety)
  [DefinitionsConfig]
    [Constants]:
          [ConstantDefinition](ep): Type='Real'
          [ConstantDefinition](W): Type='Real'
          [ConstantDefinition](V): Type='Real'
    [Functions]:
          [FunctionDefinition](stopDist): Returns='Real', Params=(Real v), Body='0'
          [FunctionDefinition](accelComp): Returns='Real', Params=(Real v), Body='ep*(v+V)'
          [FunctionDefinition](admissibleSeparation): Returns='Real', Params=(Real v), Body='stopDist(v) + accelComp(v)'
    [Predicates]:
          [PredicateDefinition](isWellformedDir): Params=(Real dx, Real dy), Body='dx^2 + dy^2 = 1'
          [PredicateDefinition](bounds): Params=(), Body='ep > 0
              & V >= 0'
          [PredicateDefinition](initialState): Params=(

In [ ]:
import gymnasium as gym
# The class name is derived from the Theorem name: "IJRR17/Theorem 1: Static safety"
# spaces, slashes, colons, etc., are replaced with underscores.
from generated_kx_env import IJRR17_Corollary_1_1_Velocity_controlled_passive_safety
import numpy as np

# Instantiate the environment
env = IJRR17_Corollary_1_1_Velocity_controlled_passive_safety()

observation, info = env.reset()
print("Initial observation:", observation)
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

# Example 1: Choose the first branch: Obstacle movement (index 0 for ChoiceLevel_0)
# Requires 'assign_vxo' and 'assign_vyo'
action_branch_0 = {
    'choice_ChoiceLevel_0': 0, # Select the obstacle movement branch (index 0)
    'assign_vxo': 0.1,         # Example value for vxo (constrained by V)
    'assign_vyo': 0.2,         # Example value for vyo (constrained by V)
    'choice_ChoiceLevel_1': 0, # Dummy value for unused nested choice
    'assign_v': 0.0,           # Dummy value for unused nondeterministic assignments
    'assign_w': 0.0,
    'assign_r': 0.0,
    'assign_xo': 0.0,
    'assign_yo': 0.0,
}

print("\nTaking 5 steps with action_branch_0 (obstacle movement):")
for i in range(5):
    observation, reward, terminated, truncated, info = env.step(action_branch_0)
    print(f"Step {i+1}: Obs={observation}, Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break
env.close()

# Example 2: Reset and choose the agent movement branch (index 1 for ChoiceLevel_0)
# Then, within agent movement, choose the 'agent moves' sub-branch (index 1 for ChoiceLevel_1)
print("\nResetting environment and taking 5 steps with action_branch_2 (agent moves):")
observation, info = env.reset()

action_branch_2 = {
    'choice_ChoiceLevel_0': 1, # Select the agent movement branch (index 1)
    'choice_ChoiceLevel_1': 1, # Select the agent *moving* sub-branch (index 1 within ChoiceLevel_1)
    'assign_vxo': 0.0,         # Dummy value for unused obstacle assignment
    'assign_vyo': 0.0,         # Dummy value for unused obstacle assignment
    'assign_v': 1.0,           # Example value for v (must be >= 0)
    'assign_w': 0.1,           # Example value for w (within -W to W, assuming W=0.5)
    'assign_r': 10.0,          # Example value for r (r != 0 and r*w == v) - make sure this holds!
                               # Here: 10.0 * 0.1 = 1.0, so r*w == v is satisfied.
    'assign_xo': 10.0,         # Example value for xo
    'assign_yo': 0.0,          # Example value for yo
}

for i in range(5):
    observation, reward, terminated, truncated, info = env.step(action_branch_2)
    print(f"Step {i+1}: Obs={observation}, Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break

env.close()

# Example 3: Reset and choose the agent movement branch (index 1 for ChoiceLevel_0)
# Then, within agent movement, choose the 'agent stops' sub-branch (index 0 for ChoiceLevel_1)
print("\nResetting environment and taking 5 steps with action_branch_3 (agent stops):")
observation, info = env.reset()

action_branch_3 = {
    'choice_ChoiceLevel_0': 1, # Select the agent movement branch (index 1)
    'choice_ChoiceLevel_1': 0, # Select the agent *stops* sub-branch (index 0 within ChoiceLevel_1)
    'assign_vxo': 0.0,         # Dummy value for unused obstacle assignment
    'assign_vyo': 0.0,         # Dummy value for unused obstacle assignment
    'assign_v': 0.0,           # Dummy value, as v will be set to 0 by deterministic assignment
    'assign_w': 0.0,           # Dummy value, as w will be set to 0 by deterministic assignment
    'assign_r': 0.0,           # Dummy value for unused nondeterministic assignments
    'assign_xo': 0.0,
    'assign_yo': 0.0,
}

for i in range(5):
    observation, reward, terminated, truncated, info = env.step(action_branch_3)
    print(f"Step {i+1}: Obs={observation}, Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break

env.close()

Initial observation: [0. 0. 1. 0. 1. 0. 0. 1. 5. 0. 0. 0. 0.]
Observation Space: Box(-inf, inf, (13,), float32)
Action Space: Dict('assign_r': Box(-inf, inf, (), float32), 'assign_v': Box(-inf, inf, (), float32), 'assign_vxo': Box(-inf, inf, (), float32), 'assign_vyo': Box(-inf, inf, (), float32), 'assign_w': Box(-inf, inf, (), float32), 'assign_xo': Box(-inf, inf, (), float32), 'assign_yo': Box(-inf, inf, (), float32), 'choice_ChoiceLevel_0': Discrete(2), 'choice_ChoiceLevel_1': Discrete(2))

Taking 5 steps with action_branch_0 (a := -b;):
Step 1: Obs=[0. 0. 1. 0. 1. 0. 0. 1. 5. 0. 0. 0. 0.], Reward=-50.00, Terminated=True, Truncated=False
Episode finished.

Resetting environment and taking 5 steps with action_branch_2 (a := A; w := *; etc.):


IndexError: list index out of range

#Smart agent

In [4]:
import gymnasium as gym
from ideal_generated_kx_env import IJRR17_Theorem_1__Static_safety
import numpy as np
import random # For the agent's random choices

# --- SimpleKXAgent Definition ---
class SimpleKXAgent:
    """
    A heuristic agent for the IJRR17_Theorem_1__Static_safety environment.
    It attempts to satisfy the complex constraints of Branch 2, falling back
    to Branch 0 if conditions cannot be met.
    """
    def __init__(self, env):
        self.env = env
        # Store the environment's constants for use in action generation.
        # These should be set in the environment before the agent acts.
        # Defaults are provided in case they are not set in the environment.
        self.W = self.env.constants.get('W', 1.0)
        self.A = self.env.constants.get('A', 1.0)
        self.b = self.env.constants.get('b', 0.5)
        self.ep = self.env.constants.get('ep', 1.0)
        # Get dt from the environment, defaulting to 1.0 if not found (common for KxEnvs)
        self.dt = getattr(self.env, 'dt', 1.0)

    def _calculate_admissible_separation(self, v_val):
        """
        Calculates admissibleSeparation(v) based on environment constants.
        Uses absolute value of velocity for safety distance components.
        """
        # Use absolute value of velocity for safety distance calculation
        v_speed = abs(v_val) 

        b_val = self.b if abs(self.b) > 1e-6 else 0.01 # Safeguard against division by zero

        stop_dist_val = (v_speed**2) / (2 * b_val)
        # For safety, the reaction time component should generally add to the distance,
        # so using v_speed here is more robust than v_val (which could be negative).
        accel_comp_val = (self.A / b_val + 1) * (self.A / 2 * self.ep**2 + self.ep * v_speed)
        admissible_separation_val = stop_dist_val + accel_comp_val
        return admissible_separation_val

    def get_action(self, observation, env_state):
        """
        Decides on an action based on the current environment state.
        Attempts to satisfy Branch 2 constraints, otherwise falls back to Branch 0.
        Crucially, it predicts the next state for constraint checking.
        """
        # Default action dictionary with dummy values for nondeterministic assignments
        action = {
            'choice_ChoiceLevel_0': 0, # Default to Branch 0
            'assign_w': 0.0,
            'assign_r': 1.0,
            'assign_xo': 0.0,
            'assign_yo': 0.0,
        }

        # Current state values
        x_current = env_state['x']
        y_current = env_state['y']
        v_current = env_state['v']

        # --- IMPORTANT CORRECTION ---
        # The environment's ActionConstraintConfig for Branch 2, specifically:
        # `? abs(x-xo) > admissibleSeparation(v) | abs(y-yo) > admissibleSeparation(v)`
        # evaluates `x`, `y`, and `v` based on the state *after* all discrete assignments
        # in the current step, but *before* the ODE integration.
        # Therefore, `x`, `y`, and `v` in this constraint refer to `x_current`, `y_current`, `v_current`.
        # The agent's previous logic incorrectly calculated `admissibleSeparation` based on `v_future`
        # and separation from `x_future`, `y_future`.

        # Calculate admissibleSeparation(v_current) for the immediate constraint check
        # This is the value that the environment will use for the constraint.
        admissible_separation_val_for_constraint = self._calculate_admissible_separation(v_current)
        
        # --- Attempt to take Branch 2 ---
        action['choice_ChoiceLevel_0'] = 2

        # --- Handle r*w = v and -W <= w <= W ---
        # This part correctly uses v_current.
        if abs(v_current) < 1e-6: # v_current is effectively zero
            action['assign_w'] = 0.0
            # If v is zero, r can be anything non-zero. Let's pick a positive r.
            action['assign_r'] = random.uniform(0.5, 5.0) # Ensure r is positive and non-zero
        else: # v_current is not zero
            W_val = self.W if abs(self.W) > 1e-6 else 0.1 # Safeguard against W being zero
            min_abs_r = abs(v_current) / W_val
            
            # Pick a magnitude for r, ensuring it's large enough and non-zero.
            # Ensure r_magnitude is at least 0.1 to avoid very small r values that could lead to huge w.
            # We'll keep r positive for simplicity.
            r_magnitude = random.uniform(max(min_abs_r + 0.1, 0.1), min_abs_r + 5.0)
            action['assign_r'] = r_magnitude # Keep r positive
            
            # Calculate w based on chosen r and current v
            action['assign_w'] = v_current / action['assign_r']
            
            # Re-check if w is within [-W, W] (should be if r was picked correctly)
            if not (-self.W <= action['assign_w'] <= self.W):
                # This should ideally not happen if min_abs_r logic is correct.
                # If it does, it implies a numerical issue or a very tight bound.
                # For robustness, we fall back.
                print(f"Agent Warning: Calculated w={action['assign_w']:.2f} is outside [-{self.W:.2f}, {self.W:.2f}]. Falling back to Branch 0.")
                action['choice_ChoiceLevel_0'] = 0
                return action # Return the fallback action immediately

        # --- Handle safety constraint: abs(x-xo) > admissibleSeparation(v) | abs(y-yo) > admissibleSeparation(v) ---
        # We need to ensure that the *current* x, y are sufficiently separated from xo, yo.
        # This uses x_current, y_current, and the `admissible_separation_val_for_constraint` calculated above.
        safety_margin = admissible_separation_val_for_constraint + 0.1 # Add a small buffer for robustness

        # Randomly decide to make x-distance or y-distance large enough
        if random.random() > 0.5:
            # Ensure |x_current - xo| > safety_margin
            action['assign_xo'] = x_current + random.choice([-1, 1]) * (safety_margin + random.uniform(0.1, 1.0))
            action['assign_yo'] = y_current + random.uniform(-1.0, 1.0) # yo can be anything relative to y_current
        else:
            # Ensure |y_current - yo| > safety_margin
            action['assign_yo'] = y_current + random.choice([-1, 1]) * (safety_margin + random.uniform(0.1, 1.0))
            action['assign_xo'] = x_current + random.uniform(-1.0, 1.0) # xo can be anything relative to x_current

        return action

# --- Environment Setup and Testing ---

# Instantiate the environment
env = IJRR17_Theorem_1__Static_safety()

# --- Initial Environment Info ---
observation, info = env.reset()
print("--- Environment Information ---")
print(f"Initial observation: {observation}")
print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")
print(f"Initial State Info: {env._get_info()}")

# Set constants after reset, as the environment might re-initialize them
env.constants['W'] = 1.0
env.constants['A'] = 1.0
env.constants['b'] = 0.5
env.constants['ep'] = 1.0
print(f"Environment Constants Set: W={env.constants['W']}, A={env.constants['A']}, b={env.constants['b']}, ep={env.constants['ep']}")


# Example 1: Choose the first branch: { a := -b; } (index 0)
print("\n--- User's Test: Taking 5 steps with action_branch_0 (a := -b;) ---")
# Reset environment before this test block to ensure clean state
observation, info = env.reset() 
env.constants['W'] = 1.0
env.constants['A'] = 1.0
env.constants['b'] = 0.5
env.constants['ep'] = 1.0
print(f"Initial observation: {observation}, Initial state: {env._get_info()}")

for i in range(5):
    action_branch_0 = {
        'choice_ChoiceLevel_0': 0, # Select the first branch
        'assign_w': 0.0,           # Dummy value for unused nondeterministic assignments
        'assign_r': 0.0,
        'assign_xo': 0.0,
        'assign_yo': 0.0,
    }
    print(f"Agent chose action: {action_branch_0}")
    observation, reward, terminated, truncated, info = env.step(action_branch_0)
    print(f"Step {i+1}: Obs={observation}, Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break
env.close()

# Example 2: Reset and choose the third branch: { a := A; w := *; ... } (index 2)
print("\n--- User's Test: Resetting environment and taking 5 steps with action_branch_2 (a := A; w := *; etc.) ---")
# Re-instantiate for a fresh start or reset
env = IJRR17_Theorem_1__Static_safety() 
observation, info = env.reset()

# Re-set constants after reset
env.constants['W'] = 1.0
env.constants['A'] = 1.0
env.constants['b'] = 0.5
env.constants['ep'] = 1.0
print(f"Initial observation: {observation}, Initial state: {env._get_info()}")

# Define action_branch_2
# It's crucial that r != 0 and r*w = v for the constraint to pass.
# Let's assume v is initially 1.0 (from reset).
# If w = 0.1, then r must be 1.0 / 0.1 = 10.0
# For this example, we need to ensure the safety constraint is met for the *current* state.
# Initial state: x=0, y=0, v=1.0. dt=1.0 (assumed). A=1.0.
# admissibleSeparation(v=1.0):
# stop_dist = (1**2)/(2*0.5) = 1.0
# accel_comp = (1/0.5 + 1) * (1/2 * 1**2 + 1 * 1) = 3 * (0.5 + 1) = 3 * 1.5 = 4.5
# total = 1.0 + 4.5 = 5.5
# So, we need abs(x_current - xo) > 5.5 or abs(y_current - yo) > 5.5.
# Let's set xo far from x_current=0.0
action_branch_2 = {
    'choice_ChoiceLevel_0': 2, # Select the third branch
    'assign_w': 0.1,           # Example value for w (within -W to W, i.e., -1.0 to 1.0)
    'assign_r': 10.0,          # Example value for r (r!=0 and r*w = v, so 10.0 * 0.1 = 1.0)
    'assign_xo': 0.0 + 6.0,    # x_current (0.0) + buffer (6.0 > 5.5)
    'assign_yo': 0.0,          # Example value for yo
}

for i in range(5):
    print(f"Agent chose action: {action_branch_2}")
    observation, reward, terminated, truncated, info = env.step(action_branch_2)
    print(f"Step {i+1}: Obs={observation}, Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break
env.close()


# --- Testing with SimpleKXAgent ---
print("\n--- Testing with SimpleKXAgent ---")
env = IJRR17_Theorem_1__Static_safety() # Re-instantiate for a clean start
agent = SimpleKXAgent(env) # Instantiate the agent

num_episodes = 10
max_steps_per_episode = 100
total_successful_episodes = 0
total_steps_taken = 0

for episode in range(num_episodes):
    print(f"\n--- Episode {episode + 1}/{num_episodes} ---")
    observation, info = env.reset()
    
    # Set constants after reset, as the environment might re-initialize them.
    # This is crucial for the agent's calculations to be consistent with the env.
    env.constants['W'] = 1.0
    env.constants['A'] = 1.0
    env.constants['b'] = 0.5
    env.constants['ep'] = 1.0
    
    # Update agent's internal constants in case they changed or were not set initially
    agent.W = env.constants['W']
    agent.A = env.constants['A']
    agent.b = env.constants['b']
    agent.ep = env.constants['ep']
    agent.dt = getattr(env, 'dt', 1.0) # Ensure agent has the correct dt

    print(f"Initial Obs: {np.round(observation, 2)}, Initial State: {env._get_info()}")
    
    episode_steps = 0
    for step in range(max_steps_per_episode):
        # Agent decides on an action based on current observation and internal state
        action = agent.get_action(observation, env._get_info()) # Pass env.state (via _get_info)
        
        print(f"  Step {step + 1}: Agent chose action: {action}")
        observation, reward, terminated, truncated, info = env.step(action)
        episode_steps += 1
        total_steps_taken += 1
        
        print(f"    Obs: {np.round(observation, 2)}, Reward: {reward:.2f}, Terminated: {terminated}, Truncated: {truncated}")
        
        if terminated or truncated:
            print(f"  Episode {episode + 1} finished at step {step + 1}. Reason: {'Terminated' if terminated else 'Truncated'}.")
            if not terminated and not truncated: # If it reached max steps without terminating/truncating
                 total_successful_episodes += 1
            break
    else: # This else block executes if the loop completes without a 'break'
        print(f"  Episode {episode + 1} completed {max_steps_per_episode} steps without terminating or truncating.")
        total_successful_episodes += 1

env.close()

print("\n--- Simulation Summary ---")
print(f"Total Episodes Run: {num_episodes}")
print(f"Episodes Completed (not terminated/truncated within max_steps): {total_successful_episodes}")
print(f"Total Steps Across All Episodes: {total_steps_taken}")

--- Environment Information ---
Initial observation: [0. 0. 1. 0. 1. 0. 0. 1. 5. 0. 0.]
Observation Space: Box(-inf, inf, (11,), float32)
Action Space: Dict('assign_r': Box(-inf, inf, (), float32), 'assign_w': Box(-inf, inf, (), float32), 'assign_xo': Box(-inf, inf, (), float32), 'assign_yo': Box(-inf, inf, (), float32), 'choice_ChoiceLevel_0': Discrete(3))
Initial State Info: {'x': 0.0, 'y': 0.0, 'v': 1.0, 'a': 0.0, 'dx': np.float64(1.0), 'dy': np.float64(0.0), 'w': 0.0, 'r': 1.0, 'xo': 5.0, 'yo': 0.0, 't': 0.0}
Environment Constants Set: W=1.0, A=1.0, b=0.5, ep=1.0

--- User's Test: Taking 5 steps with action_branch_0 (a := -b;) ---
Initial observation: [0. 0. 1. 0. 1. 0. 0. 1. 5. 0. 0.], Initial state: {'x': 0.0, 'y': 0.0, 'v': 1.0, 'a': 0.0, 'dx': np.float64(1.0), 'dy': np.float64(0.0), 'w': 0.0, 'r': 1.0, 'xo': 5.0, 'yo': 0.0, 't': 0.0}
Agent chose action: {'choice_ChoiceLevel_0': 0, 'assign_w': 0.0, 'assign_r': 0.0, 'assign_xo': 0.0, 'assign_yo': 0.0}
--- Step 1 ---
Action: {'cho

#Clueless agent

In [8]:
import gymnasium as gym
from gymnasium import spaces # Import spaces for type checking
from ideal_generated_kx_env import IJRR17_Theorem_1__Static_safety
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical, Normal
import os
import matplotlib.pyplot as plt # For plotting learning progress

# --- Constants for training ---
GAMMA = 0.99  # Discount factor for rewards
LR_ACTOR = 0.0005 # Learning rate for the actor
LR_CRITIC = 0.001 # Learning rate for the critic
CLIP_GRAD_NORM = 0.5 # Gradient clipping to prevent exploding gradients

# --- Neural Network Definitions ---
class Actor(nn.Module):
    def __init__(self, obs_dim, discrete_action_dim, continuous_action_dim):
        super(Actor, self).__init__()
        self.fc1 = nn.Linear(obs_dim, 128)
        self.fc2 = nn.Linear(128, 128)

        self.discrete_action_dim = discrete_action_dim
        self.continuous_action_dim = continuous_action_dim

        # Conditionally create discrete head
        if self.discrete_action_dim > 0:
            self.discrete_head = nn.Linear(128, self.discrete_action_dim)
        else:
            self.discrete_head = None

        # Conditionally create continuous heads
        if self.continuous_action_dim > 0:
            self.continuous_mean_head = nn.Linear(128, self.continuous_action_dim)
            # Initialize log_std to a small negative value
            self.continuous_log_std_head = nn.Parameter(torch.zeros(1, self.continuous_action_dim))
        else:
            self.continuous_mean_head = None
            self.continuous_log_std_head = None

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))

        discrete_logits = None
        continuous_means = None
        continuous_log_stds = None

        if self.discrete_head is not None:
            discrete_logits = self.discrete_head(x)

        if self.continuous_mean_head is not None:
            continuous_means = self.continuous_mean_head(x)
            # Expand log_std to match the batch size and action dimension of means.
            continuous_log_stds = self.continuous_log_std_head.expand_as(continuous_means).clamp(-20, 2)
        
        return discrete_logits, continuous_means, continuous_log_stds

class Critic(nn.Module):
    def __init__(self, obs_dim):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(obs_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.value_head = nn.Linear(128, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.value_head(x)

# --- Learning Agent Definition (A2C-like) ---
class LearningAgent:
    def __init__(self, env, model_dir="models"):
        self.env = env
        self.model_dir = model_dir
        os.makedirs(self.model_dir, exist_ok=True)

        self.obs_dim = env.observation_space.shape[0]

        # Dynamically determine discrete and continuous action dimensions and their keys
        self.discrete_action_dim = 0
        self.continuous_action_dim = 0
        self.discrete_action_key = None
        self.continuous_action_keys = []
        self.continuous_action_component_shapes = {} # To store original shapes for reconstruction

        if isinstance(env.action_space, spaces.Dict):
            for key, space in env.action_space.items():
                if isinstance(space, spaces.Discrete):
                    if self.discrete_action_key is not None:
                        print(f"Warning: Multiple discrete action spaces found. Using the first one identified ('{self.discrete_action_key}').")
                    else:
                        self.discrete_action_dim = space.n
                        self.discrete_action_key = key
                elif isinstance(space, spaces.Box):
                    # Handle scalar Box spaces (shape=()) correctly
                    self.continuous_action_dim += space.shape[0] if space.shape else 1
                    self.continuous_action_component_shapes[key] = space.shape[0] if space.shape else 1
                    self.continuous_action_keys.append(key)
            
            if self.discrete_action_key is None:
                print("Info: No discrete action space found in the environment's action_space Dict.")
            if not self.continuous_action_keys:
                print("Info: No continuous action spaces found in the environment's action_space Dict.")
        else:
            raise TypeError(f"Environment action space is of type {type(env.action_space)}, but LearningAgent expects a gymnasium.spaces.Dict space.")

        if self.discrete_action_dim == 0 and self.continuous_action_dim == 0:
            raise ValueError("Could not determine any action dimensions from the environment's action space. Check its definition.")

        self.actor = Actor(self.obs_dim, self.discrete_action_dim, self.continuous_action_dim)
        self.critic = Critic(self.obs_dim)

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=LR_ACTOR)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=LR_CRITIC)

        self.log_probs = []
        self.values = []
        self.rewards = []
        self.masks = []

    def get_action(self, observation):
        obs_tensor = torch.FloatTensor(observation).unsqueeze(0)
        discrete_logits, continuous_means, continuous_log_stds = self.actor(obs_tensor)

        action = {}
        # FIX: Initialize total_log_prob as a tensor with a batch dimension (1,)
        total_log_prob = torch.zeros(1, device=obs_tensor.device) 

        # --- Discrete action sampling ---
        if self.discrete_action_key is not None:
            discrete_dist = Categorical(logits=discrete_logits)
            discrete_action = discrete_dist.sample() # Shape (1,)
            action[self.discrete_action_key] = discrete_action.item()
            total_log_prob += discrete_dist.log_prob(discrete_action) # Shape (1,)

        # --- Continuous action sampling ---
        if self.continuous_action_keys:
            continuous_stds = torch.exp(continuous_log_stds)
            continuous_dist = Normal(continuous_means, continuous_stds)
            continuous_actions_tensor = continuous_dist.sample() # Shape: (1, self.continuous_action_dim)
            
            # Calculate log-probability for continuous actions. Sum over the action dimensions.
            total_log_prob += continuous_dist.log_prob(continuous_actions_tensor).sum(dim=-1) # Shape (1,)

            # Assign continuous actions back to their original keys
            current_idx = 0
            for key in self.continuous_action_keys:
                component_size = self.continuous_action_component_shapes[key]
                component_value_tensor = continuous_actions_tensor[0, current_idx : current_idx + component_size]
                
                if component_size == 1:
                    action[key] = np.float64(component_value_tensor.item())
                else:
                    # If a Box space was shape=(N,), then it would be an array.
                    action[key] = component_value_tensor.cpu().numpy().astype(np.float64)
                
                current_idx += component_size
        
        value = self.critic(obs_tensor)
        self.log_probs.append(total_log_prob)
        self.values.append(value)
        
        return action

    def store_transition(self, reward, terminated, truncated):
        self.rewards.append(reward)
        self.masks.append(1.0 - (terminated or truncated))

    def learn(self, next_observation):
        if not self.rewards: # No steps taken in episode, nothing to learn
            return

        log_probs = torch.cat(self.log_probs)
        values = torch.cat(self.values)
        rewards = torch.FloatTensor(self.rewards)
        masks = torch.FloatTensor(self.masks)

        next_obs_tensor = torch.FloatTensor(next_observation).unsqueeze(0)
        next_value = self.critic(next_obs_tensor).detach()

        returns = torch.zeros_like(rewards)
        R = next_value * masks[-1]

        for t in reversed(range(len(rewards))):
            R = rewards[t] + GAMMA * R * masks[t]
            returns[t] = R

        advantages = returns - values.squeeze()

        # --- Actor Loss ---
        actor_loss = -(log_probs * advantages.detach()).mean()
        
        # --- Critic Loss ---
        critic_loss = advantages.pow(2).mean()

        # --- Update Actor Network ---
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), CLIP_GRAD_NORM)
        self.actor_optimizer.step()

        # --- Update Critic Network ---
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), CLIP_GRAD_NORM)
        self.critic_optimizer.step()

        # Clear buffers
        self.log_probs = []
        self.values = []
        self.rewards = []
        self.masks = []

    def save_weights(self, episode_num=None):
        actor_path = os.path.join(self.model_dir, f"actor_weights_ep{episode_num}.pth" if episode_num else "actor_weights_final.pth")
        critic_path = os.path.join(self.model_dir, f"critic_weights_ep{episode_num}.pth" if episode_num else "critic_weights_final.pth")
        torch.save(self.actor.state_dict(), actor_path)
        torch.save(self.critic.state_dict(), critic_path)
        print(f"Saved actor weights to {actor_path}")
        print(f"Saved critic weights to {critic_path}")

    def load_weights(self, episode_num=None):
        actor_path = os.path.join(self.model_dir, f"actor_weights_ep{episode_num}.pth" if episode_num else "actor_weights_final.pth")
        critic_path = os.path.join(self.model_dir, f"critic_weights_ep{episode_num}.pth" if episode_num else "critic_weights_final.pth")
        
        if os.path.exists(actor_path) and os.path.exists(critic_path):
            self.actor.load_state_dict(torch.load(actor_path))
            self.critic.load_state_dict(torch.load(critic_path))
            print(f"Loaded weights from episode {episode_num if episode_num else 'final'}.")
            return True
        else:
            print(f"No weights found at {actor_path} and {critic_path}. Starting from scratch.")
            return False

# --- Environment Setup and Testing ---

# Instantiate the environment for initial info
env = IJRR17_Theorem_1__Static_safety()

# --- Initial Environment Information ---
observation, info = env.reset()
print("--- Environment Information ---")
print(f"Initial observation: {observation}")
print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}") # This will now show the actual action space structure
print(f"Initial State Info: {env._get_info()}")

# Set constants after reset. These are environment-specific and not learned by the agent.
env.constants['W'] = 1.0
env.constants['A'] = 1.0
env.constants['b'] = 0.5
env.constants['ep'] = 1.0
print(f"Environment Constants Set: W={env.constants['W']}, A={env.constants['A']}, b={env.constants['b']}, ep={env.constants['ep']}")
env.close() # Close the initial env instance

# --- Training / Testing with LearningAgent ---
print("\n--- Training with LearningAgent ---")

# --- Configuration for Training ---
num_episodes = 2000          # Total number of episodes to run
max_steps_per_episode = 100  # Maximum steps before an episode is truncated
SAVE_INTERVAL = 200          # Save weights every X episodes
LOAD_EXISTING_WEIGHTS = False # Set to True to resume training from previously saved 'final' weights
EVALUATE_ONLY = False        # Set to True to run without learning, using loaded weights (if any)

# Re-instantiate the environment and the learning agent
env = IJRR17_Theorem_1__Static_safety() 
agent = LearningAgent(env) 

# Load weights if configured to do so
if LOAD_EXISTING_WEIGHTS or EVALUATE_ONLY:
    agent.load_weights()

total_steps_taken = 0
episode_rewards = [] # To track learning progress over episodes

for episode in range(num_episodes):
    observation, info = env.reset()
    
    # Re-set constants after reset.
    env.constants['W'] = 1.0
    env.constants['A'] = 1.0
    env.constants['b'] = 0.5
    env.constants['ep'] = 1.0
    
    episode_reward = 0.0
    episode_steps = 0
    
    for step in range(max_steps_per_episode):
        action = agent.get_action(observation) 
        
        next_observation, reward, terminated, truncated, info = env.step(action)
        
        if not EVALUATE_ONLY:
            agent.store_transition(reward, terminated, truncated)
        
        observation = next_observation
        episode_reward += reward
        episode_steps += 1
        total_steps_taken += 1
        
        if terminated or truncated:
            break
    
    if not EVALUATE_ONLY:
        agent.learn(observation) 
    
    episode_rewards.append(episode_reward)
    
    avg_reward_last_100 = np.mean(episode_rewards[-100:])
    print(f"Episode {episode + 1}/{num_episodes}, Steps: {episode_steps}, Total Reward: {episode_reward:.2f}, Avg Reward (last 100): {avg_reward_last_100:.2f}")

    if (episode + 1) % SAVE_INTERVAL == 0 and not EVALUATE_ONLY:
        agent.save_weights(episode + 1)

if not EVALUATE_ONLY:
    agent.save_weights()

env.close()

print("\n--- Simulation Summary ---")
print(f"Total Episodes Run: {num_episodes}")
print(f"Total Steps Across All Episodes: {total_steps_taken}")
if episode_rewards:
    print(f"Average Reward over last 100 episodes: {np.mean(episode_rewards[-100:]):.2f}")
    print(f"Max Reward achieved: {np.max(episode_rewards):.2f}")

if episode_rewards:
    plt.figure(figsize=(12, 6))
    plt.plot(episode_rewards)
    plt.title('Episode Rewards Over Time')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.grid(True)
    plt.show()

--- Environment Information ---
Initial observation: [0. 0. 1. 0. 1. 0. 0. 1. 5. 0. 0.]
Observation Space: Box(-inf, inf, (11,), float32)
Action Space: Dict('assign_r': Box(-inf, inf, (), float32), 'assign_w': Box(-inf, inf, (), float32), 'assign_xo': Box(-inf, inf, (), float32), 'assign_yo': Box(-inf, inf, (), float32), 'choice_ChoiceLevel_0': Discrete(3))
Initial State Info: {'x': 0.0, 'y': 0.0, 'v': 1.0, 'a': 0.0, 'dx': np.float64(1.0), 'dy': np.float64(0.0), 'w': 0.0, 'r': 1.0, 'xo': 5.0, 'yo': 0.0, 't': 0.0}
Environment Constants Set: W=1.0, A=1.0, b=0.5, ep=1.0

--- Training with LearningAgent ---
--- Step 1 ---
Action: {'choice_ChoiceLevel_0': 0, 'assign_r': np.float64(-1.0382359027862549), 'assign_w': np.float64(-0.32951587438583374), 'assign_xo': np.float64(2.0996456146240234), 'assign_yo': np.float64(1.6349908113479614)}
State before control: {'x': 0.0, 'y': 0.0, 'v': 1.0, 'a': 0.0, 'dx': np.float64(1.0), 'dy': np.float64(0.0), 'w': 0.0, 'r': 1.0, 'xo': 5.0, 'yo': 0.0, 't': 0

KeyboardInterrupt: 